# CP1 — Data Ingestion: Amazon Product Dataset 2020 → Household-Cleaning slice

**Owner:** Shane · **Deliverable:** Checkpoint 1 ingestion notebook

This notebook curates a **Household Cleaning** slice of the *Amazon Product Dataset 2020* (Kaggle) and writes two tidy tables used by the rest of the pipeline:

| file | schema |
|---|---|
| `data/processed/products.parquet` | `doc_id, sku, asin, title, brand, category, price, list_price, rating, features, ingredients, size_oz, price_per_oz, url, stock` |
| `data/processed/reviews.parquet`  | `product_id, stars, snippet` |

It runs **unchanged** on the shipped 24-row `SAMPLE_...csv` and on the real Kaggle file — point `RAW_CSV` at whichever you have.

> The heavy lifting lives in `src/rag/ingest.py` and `src/rag/schema.py` so the > exact same logic is reused by the build script and the MCP tool. The cells > below both *call* that module and *show* the key transformations for review.

## 0. Setup

In [1]:
import sys, os
sys.path.append(os.path.abspath('../src'))
import pandas as pd
pd.set_option('display.max_colwidth', 60)

from rag.config import get_config
from rag import ingest, schema

cfg = get_config()
# To use the REAL Kaggle file instead of the sample, uncomment and edit:
# cfg.raw_csv = '../data/raw/marketing_sample_for_amazon_com-ecommerce__20200101_20200131__10k_data.csv'
print('Raw CSV :', cfg.raw_csv)
print('Output  :', cfg.processed_dir)

Raw CSV : /home/claude/voice-commerce-assistant/data/raw/SAMPLE_amazon_household_cleaning.csv
Output  : /home/claude/voice-commerce-assistant/data/processed


## 1. Load raw data
We read everything as strings (the Kaggle file mixes types and has missing values) and inspect the schema.

In [2]:
raw = ingest.load_raw(cfg.raw_csv)
print('raw shape:', raw.shape)
print('columns:', list(raw.columns))
raw.head(3)

raw shape: (24, 30)
columns: ['Uniq Id', 'Product Name', 'Brand Name', 'Asin', 'Category', 'Upc Ean Code', 'List Price', 'Selling Price', 'Quantity', 'Model Number', 'About Product', 'Product Specification', 'Technical Details', 'Shipping Weight', 'Product Dimensions', 'Image', 'Variants', 'Sku', 'Product Url', 'Stock', 'Product Details', 'Dimensions', 'Color', 'Ingredients', 'Direction To Use', 'Is Amazon Seller', 'Size Quantity Variant', 'Product Description', 'Rating', 'Review Snippets']


                            Uniq Id  \
0  74d9f6149b125affab1f3b8d14798b0b   
1  c290f2c3fcf0554d94d625b7a77820d9   
2  79270c8c173d555181a8e845b143d93c   

                                             Product Name   Brand Name  \
0  Steel-Safe Eco Stainless Steel Cleaner & Polish, 16 oz   GreenGleam   
1    Brushed Metal Miracle Stainless Cleaner Spray, 12 oz     PureHome   
2         ProShine Stainless Steel Polish, Aerosol, 15 oz  ShineMaster   

          Asin                                                     Category  \
0  B0SAMPLE000  Health & Household | Household Supplies | Cleaning & Hou...   
1  B0SAMPLE001  Health & Household | Household Supplies | Cleaning & Hou...   
2  B0SAMPLE002  Health & Household | Household Supplies | Cleaning & Hou...   

  Upc Ean Code List Price Selling Price Quantity Model Number  ...  \
0                 $ 15.99       $ 12.49               GRE-000  ...   
1                 $ 13.49        $ 9.99               PUR-001  ...   
2                 $

## 2. Filter to the Household-Cleaning slice
We keep rows whose `Category` mentions any cleaning/household keyword (`cfg.slice_keywords`). On the real 10k file this narrows ~10,000 rows down to the cleaning products; on the sample it is already the slice.

In [3]:
print('slice keywords:', cfg.slice_keywords)
sliced = ingest.filter_slice(raw, cfg)
print(f'{len(sliced)} / {len(raw)} rows kept')
sliced[[ingest.COL['title'], ingest.COL['category']]].head()

slice keywords: ('clean', 'household', 'cleaning', 'deterg', 'dish', 'laundry', 'polish', 'wipe', 'disinfect', 'bath')
24 / 24 rows kept


                                             Product Name  \
0  Steel-Safe Eco Stainless Steel Cleaner & Polish, 16 oz   
1    Brushed Metal Miracle Stainless Cleaner Spray, 12 oz   
2         ProShine Stainless Steel Polish, Aerosol, 15 oz   
3       EcoBright Stainless & Chrome Cleaner Wipes, 30 ct   
4          Sparkle Naturals Glass & Window Cleaner, 26 oz   

                                                      Category  
0  Health & Household | Household Supplies | Cleaning & Hou...  
1  Health & Household | Household Supplies | Cleaning & Hou...  
2  Health & Household | Household Supplies | Cleaning & Hou...  
3  Health & Household | Household Supplies | Cleaning & Hou...  
4  Health & Household | Household Supplies | Cleaning & Hou...  

## 3. Field normalization (the interesting part)
`schema.py` normalizes the messy Amazon fields into analysis-ready values:

* **price** — `'$ 12.49'` → `12.49` (falls back from *Selling Price* to *List Price*)
* **size_oz** — parsed from *Size Quantity Variant* / title (`'16 Fl Oz'` → `16.0`, `'500 ml'` → `16.9`)
* **price_per_oz** — enables *fair* comparisons across pack sizes
* **features** — the `'|'`-separated *About Product* with the boilerplate dropped
* **rating** — from the optional *Rating* column (NaN on the real file until reviews are joined)

In [4]:
examples = ['$ 12.49', '1,234.00', '', 'nan']
print('parse_price :', [schema.parse_price(x) for x in examples])
print('parse_size_oz:', schema.parse_size_oz('16 Fl Oz'), schema.parse_size_oz('500 ml'),
      schema.parse_size_oz('1.5 lb'))
print('price_per_oz :', schema.price_per_oz(12.49, 16.0))
print('features     :', schema.split_features('Make sure this fits... | Plant-based | Streak-free'))

parse_price : [12.49, 1234.0, None, None]
parse_size_oz: 16.0 16.907 24.0
price_per_oz : 0.7806
features     : ['Plant-based', 'Streak-free']


## 4. Build the products table

In [5]:
products = ingest.build_products(sliced)
print('products shape:', products.shape)
products[['title','brand','price','size_oz','price_per_oz','rating','ingredients']].head()

products shape: (24, 15)


                                                    title             brand  \
0  Steel-Safe Eco Stainless Steel Cleaner & Polish, 16 oz        GreenGleam   
1    Brushed Metal Miracle Stainless Cleaner Spray, 12 oz          PureHome   
2         ProShine Stainless Steel Polish, Aerosol, 15 oz       ShineMaster   
3       EcoBright Stainless & Chrome Cleaner Wipes, 30 ct        GreenGleam   
4          Sparkle Naturals Glass & Window Cleaner, 26 oz  Sparkle Naturals   

   price  size_oz  price_per_oz  rating  \
0  12.49     16.0        0.7806     4.6   
1   9.99     12.0        0.8325     4.4   
2  16.75     15.0        1.1167     4.7   
3   8.49      NaN           NaN     4.3   
4   6.99     26.0        0.2688     4.5   

                                                   ingredients  
0  Water, Caprylyl/Capryl Glucoside (plant-based surfactant...  
1  Water, Decyl Glucoside, Sodium Gluconate, Citric Acid, P...  
2  Mineral oil, Aliphatic hydrocarbons, Silicone emulsion, ...  
3  Wat

## 5. Build the reviews table (optional column)

In [6]:
reviews = ingest.build_reviews(sliced)
print('reviews shape:', reviews.shape, '(empty on the real file — reviews are optional)')
reviews.head()

reviews shape: (72, 3) (empty on the real file — reviews are optional)


                         product_id  stars  \
0  74d9f6149b125affab1f3b8d14798b0b    4.6   
1  74d9f6149b125affab1f3b8d14798b0b    4.6   
2  74d9f6149b125affab1f3b8d14798b0b    4.6   
3  c290f2c3fcf0554d94d625b7a77820d9    4.4   
4  c290f2c3fcf0554d94d625b7a77820d9    4.4   

                                        snippet  
0  Best stainless cleaner I've used, no streaks  
1  Smells great and cuts fingerprints instantly  
2  Eco-friendly and actually works on my fridge  
3                         Great value under $10  
4  Works but needs a second pass on heavy grime  

## 6. Quick EDA / data-quality checks
Sanity-check coverage of the fields the retriever and answerer depend on.

In [7]:
def coverage(df):
    return pd.Series({
        'rows': len(df),
        'has_price': df['price'].notna().mean(),
        'has_rating': df['rating'].notna().mean(),
        'has_size_oz': df['size_oz'].notna().mean(),
        'has_ingredients': (df['ingredients'].str.len() > 0).mean(),
        'has_features': (df['features'].str.len() > 0).mean(),
    })
coverage(products)

rows               24.000000
has_price           1.000000
has_rating          1.000000
has_size_oz         0.833333
has_ingredients     1.000000
has_features        1.000000
dtype: float64

In [8]:
# Price distribution and the demo-relevant sub-slice: eco stainless steel cleaners < $15
print(products['price'].describe()[['min','50%','max']].round(2).to_dict())
mask = (products['title'].str.contains('stainless', case=False) &
        products['ingredients'].str.contains('plant|coco|citr|glucoside', case=False) &
        (products['price'] < 15))
products.loc[mask, ['title','brand','price','price_per_oz','rating']].sort_values('rating', ascending=False)

{'min': 3.99, '50%': 9.24, 'max': 16.75}


                                                     title       brand  price  \
0   Steel-Safe Eco Stainless Steel Cleaner & Polish, 16 oz  GreenGleam  12.49   
12           EverGreen Stainless Steel Wipes Refill, 40 ct   EverGreen  13.49   
1     Brushed Metal Miracle Stainless Cleaner Spray, 12 oz    PureHome   9.99   
18         NatureNest Stainless Steel Cleaner, 8 oz travel  NatureNest   6.49   
3        EcoBright Stainless & Chrome Cleaner Wipes, 30 ct  GreenGleam   8.49   

    price_per_oz  rating  
0         0.7806     4.6  
12           NaN     4.5  
1         0.8325     4.4  
18        0.8113     4.4  
3            NaN     4.3  

## 7. Write parquet outputs
These two files are the contract for the indexing step (`build_index.sh`).

In [9]:
paths = ingest.write_parquet(products, reviews, cfg.processed_dir)
paths

{'products': '/home/claude/voice-commerce-assistant/data/processed/products.parquet',
 'reviews': '/home/claude/voice-commerce-assistant/data/processed/reviews.parquet'}

In [10]:
# One-liner equivalent (what the build script calls):
summary = ingest.run()
summary

{'products': '/home/claude/voice-commerce-assistant/data/processed/products.parquet',
 'reviews': '/home/claude/voice-commerce-assistant/data/processed/reviews.parquet',
 'n_raw': 24,
 'n_slice': 24,
 'n_products': 24,
 'n_reviews': 72,
 'with_price': 24,
 'with_rating': 24,
 'with_ingredients': 24}

---
### Handoff notes
* **Index step** embeds `title + features + top-3 review snippets + ingredients` (`Product.embed_text`) and stores metadata (`brand, price, rating, size_oz, price_per_oz, category, ingredients, sku, url, doc_id`).
* **`rag.search`** returns `{sku, title, price, rating, brand, ingredients, doc_id}` with `url`/`score` extras for citations.
* To scale to the full Kaggle catalog, set `RAW_CSV` and (optionally) widen `cfg.slice_keywords` — nothing else changes.